# Numerical Linear Algebra: Determinants, Inverses, LU, Linear Systems, Fitting, and Interpolation

Six independent problems in numerical linear algebra, each implemented from scratch (no
`numpy.linalg` shortcuts): a recursive cofactor-expansion determinant, a Gauss-Jordan matrix
inverse, an orthogonal basis via Gram-Schmidt-style reduction, LU factorization, Gaussian
elimination for a linear system, linear vs. exponential curve fitting compared by R², Newton's
divided-difference interpolation, and Lagrange interpolation.

### 1. Matrix determinant (recursive cofactor expansion)

In [ ]:
def get_minor(matrix, i, j):
    # removes row i and column j to build the minor
    return [row[:j] + row[j+1:] for k, row in enumerate(matrix) if k != i]

def determinant(matrix):
    n = len(matrix)
    
    # base case: 1x1 matrix
    if n == 1:
        return matrix[0][0]
    
    # base case: 2x2 matrix
    if n == 2:
        return matrix[0][0]*matrix[1][1] - matrix[0][1]*matrix[1][0]
    
    det = 0
    for j in range(n):
        sign = (-1) ** j  # alternating sign
        minor = get_minor(matrix, 0, j)
        det += sign * matrix[0][j] * determinant(minor)
    
    return det

matrix = [
    [-23, -23, 16, 32, -189],
    [-31, -31, -556, 70, 682],
    [42, 96, -331, 56, -322],
    [0, 86, 69, 34, 96],
    [63, -431, -236, 62, -857]
]

print("Determinant:", determinant(matrix))


### 2. Matrix inverse (Gauss-Jordan elimination)

In [ ]:
# Ask for the number of variables
n = int(input("Give me the number of variables: "))

# Build an augmented matrix of size n x 2n (A | I)
augmented = [[0 for _ in range(2 * n)] for _ in range(n)]

# Fill in matrix A (left half of the augmented matrix)
print("Enter the coefficients of matrix A:")
for i in range(n):
    for j in range(n):
        augmented[i][j] = float(input(f"A({i+1},{j+1}): "))

# Append the identity matrix (right half of the augmented matrix)
for i in range(n):
    augmented[i][n + i] = 1.0

# Show the initial augmented matrix
print("\nInitial augmented matrix (A | Identity):")
for row in augmented:
    print([round(x, 3) for x in row])

# Gauss-Jordan method to obtain the inverse
for i in range(n):
    # Partial pivoting if the pivot is 0
    if augmented[i][i] == 0:
        for k in range(i + 1, n):
            if augmented[k][i] != 0:
                augmented[i], augmented[k] = augmented[k], augmented[i]
                break

    # Check whether the matrix is singular
    if augmented[i][i] == 0:
        print("\nThe matrix is singular and has no inverse.")
        exit()

    # Normalize row i by dividing by the pivot
    pivot = augmented[i][i]
    for j in range(2 * n):
        augmented[i][j] /= pivot

    # Zero out the other rows
    for j in range(n):
        if i != j:
            factor = augmented[j][i]
            for k in range(2 * n):
                augmented[j][k] -= factor * augmented[i][k]

# Extract the inverse matrix from the right-hand side
inverse_matrix = [[augmented[i][j] for j in range(n, 2 * n)] for i in range(n)]

# Show the inverse matrix
print("\nInverse matrix obtained:")
for row in inverse_matrix:
    print([round(x, 6) for x in row])



### 2.1 Orthogonal basis via a Gram-Schmidt-style reduction

In [ ]:

# Ask for the number of vectors (and their dimension)
n = int(input("Give me the number of vectors (and their dimension): "))

# Initialize matrices
vector_matrix = []       # original matrix, vectors as rows
vector_transpose = []    # transpose of the original matrix
dot_product = []         # dot-product matrix: V * V^T
augmented_matrix = []    # augmented matrix for reduction: [dot_product | vector_matrix]

# Build zero-filled matrices
for i in range(n):
    zero_row = [0] * n
    vector_matrix.append(zero_row[:])
    vector_transpose.append(zero_row[:])
    dot_product.append(zero_row[:])
    
    ext_row = [0] * (2 * n)
    augmented_matrix.append(ext_row)

# Request the vectors
print("\n--- Vector input ---")
for i in range(n):
    for j in range(n):
        vector_matrix[i][j] = float(input(f"Vector {i+1}, component {j+1}: "))

# Show the original matrix
print("\n--- Original matrix (vectors as rows) ---")
for row in vector_matrix:
    print(row)

# Compute the transpose
for i in range(n):
    for j in range(n):
        vector_transpose[i][j] = vector_matrix[j][i]

# Show the transposed matrix
print("\n--- Transpose of the original matrix ---")
for row in vector_transpose:
    print(row)

# Compute the dot product: V * V^T
for i in range(n):
    for j in range(n):
        total = 0
        for k in range(n):
            total += vector_matrix[i][k] * vector_transpose[k][j]
        dot_product[i][j] = total

# Show the dot-product matrix
print("\n--- Dot-product matrix V * V^T ---")
for row in dot_product:
    print(row)

# Build the augmented matrix: [dot_product | vector_matrix]
for i in range(n):
    for j in range(n):
        augmented_matrix[i][j] = dot_product[i][j]
        augmented_matrix[i][j + n] = vector_matrix[i][j]

# Show the augmented matrix before reduction
print("\n--- Augmented matrix before reduction ---")
for row in augmented_matrix:
    print(row)

# Apply Gaussian elimination to diagonalize the left-hand side (dot-product part)
for pivot in range(n - 1):
    for row in range(pivot + 1, n):
        factor = augmented_matrix[row][pivot] / augmented_matrix[pivot][pivot]
        for column in range(2 * n):
            augmented_matrix[row][column] -= factor * augmented_matrix[pivot][column]

# Show the augmented matrix after reduction
print("\n--- Augmented matrix after Gaussian elimination ---")
for row in augmented_matrix:
    print([round(elem, 6) for elem in row])

# Normalize the rows of the right-hand side (which now holds the orthogonal vectors)
for i in range(n):
    divisor = augmented_matrix[i][i]  # diagonal element
    for j in range(n):
        augmented_matrix[i][j + n] /= divisor

# Show the final matrix of orthogonal vectors
print("\n--- Orthogonal vectors (result of the Gram-Schmidt-style process) ---")
for i in range(n):
    print([round(elem, 6) for elem in augmented_matrix[i][n:]])


### 2.2 LU factorization

In [ ]:

# Ask for the number of variables (dimension of the square matrix A)
n = int(input("Give me the number of variables (n x n): "))

# Initialize matrices L, U, and the L*U product with zeros
matrix_U = []  # upper-triangular matrix
matrix_L = []  # lower-triangular matrix (1s on the diagonal)
product_LU = []  # matrix to verify L * U = A

for row in range(n):
    zero_row = [0] * n
    matrix_U.append(zero_row[:])
    matrix_L.append(zero_row[:])
    product_LU.append(zero_row[:])
    matrix_L[row][row] = 1  # L has 1s on the diagonal

# Enter the elements of matrix A to be factorized
print("\nEnter the elements of matrix A (to be factorized into L and U):")
for i in range(n):
    for j in range(n):
        matrix_U[i][j] = float(input(f"A({i+1},{j+1}): "))

# Show the original matrix A (stored in U, before it's modified)
print("\nOriginal matrix A (stored in U before modification):")
for row in matrix_U:
    print(row)

# LU factorization process (no pivoting)
for pivot in range(n - 1):  # walk the main diagonal
    for row in range(pivot + 1, n):  # rows below the pivot
        factor = matrix_U[row][pivot] / matrix_U[pivot][pivot]
        matrix_L[row][pivot] = factor  # store the factor in L
        for column in range(n):
            matrix_U[row][column] -= factor * matrix_U[pivot][column]

# Show the resulting L and U matrices
print("\nMatrix L (lower triangular):")
for row in matrix_L:
    print([round(elem, 6) for elem in row])

print("\nMatrix U (upper triangular):")
for row in matrix_U:
    print([round(elem, 6) for elem in row])

# Verify that L * U = A (matrix product)
for i in range(n):
    for j in range(n):
        total = 0
        for k in range(n):
            total += matrix_L[i][k] * matrix_U[k][j]
        product_LU[i][j] = total

# Show the L * U product
print("\nVerification of the L * U product (should equal the original matrix A):")
for row in product_LU:
    print([round(elem, 6) for elem in row])


### 3. Solving a linear system (Gaussian elimination + back-substitution)

In [ ]:
# Import required libraries
import numpy as np

# Ask for the number of variables
n = int(input("Give me the number of variables: "))

# Initialize the augmented matrix and the solution vector
augmented = [[0 for column in range(n + 1)] for row in range(n)]
solution = [0 for index in range(n)]

# Fill in the augmented matrix with the values provided by the user
for i in range(n):
    for j in range(n + 1):
        augmented[i][j] = float(input(f"Give me element ({i+1}, {j+1}): "))

# Show the initial augmented matrix
print("\nInitial augmented matrix:")
for row in augmented:
    print(row)

# Gaussian elimination process
for i in range(n - 1):
    for j in range(i + 1, n):
        factor = augmented[j][i] / augmented[i][i]
        for k in range(n + 1):
            augmented[j][k] -= factor * augmented[i][k]

# Show the augmented matrix after elimination
print("\nAugmented matrix after Gaussian elimination:")
for row in augmented:
    print(row)

# Back-substitution to find the solution
solution[n - 1] = augmented[n - 1][n] / augmented[n - 1][n - 1]

for i in range(n - 2, -1, -1):
    total = 0
    for j in range(n - 1, i, -1):
        total += augmented[i][j] * solution[j]
    solution[i] = (augmented[i][n] - total) / augmented[i][i]

# Show the final solution
print("\nSolution vector:")
for index, value in enumerate(solution):
    print(f"x{index+1} = {value:.6f}")


### 4. Linear vs. exponential curve fitting, compared by R²

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Original data
x_vals = np.array([0.05, 0.4, 0.8, 1.2, 1.6, 2, 2.4])
y_vals = np.array([550, 750, 1000, 1400, 2000, 2700, 3750])
n = len(x_vals)

# LINEAR FIT
# Model: y = a + b*x
Sx = np.sum(x_vals)
Sy = np.sum(y_vals)
Sxx = np.sum(x_vals**2)
Sxy = np.sum(x_vals * y_vals)

b_linear = (n * Sxy - Sx * Sy) / (n * Sxx - Sx**2)
a_linear = (Sy - b_linear * Sx) / n

def linear_model(x): return a_linear + b_linear * x
y_linear = linear_model(x_vals)

# R² for the linear fit
mean_y = np.mean(y_vals)
St = np.sum((y_vals - mean_y)**2)
Sr_linear = np.sum((y_vals - y_linear)**2)
R2_linear = 1 - (Sr_linear / St)

# EXPONENTIAL FIT
# Model: y = A * e^(B * x) => ln(y) = ln(A) + Bx
ln_y_vals = np.log(y_vals)
Sy_log = np.sum(ln_y_vals)
Sxy_log = np.sum(x_vals * ln_y_vals)

b_exp = (n * Sxy_log - Sx * Sy_log) / (n * Sxx - Sx**2)
a_log = (Sy_log - b_exp * Sx) / n
A = np.exp(a_log)

def exponential_model(x): return A * np.exp(b_exp * x)
y_exponential = exponential_model(x_vals)

# R² for the exponential fit
Sr_exp = np.sum((y_vals - y_exponential)**2)
R2_exp = 1 - (Sr_exp / St)

# Show the results
print("Linear model:      y = {:.3f} + {:.3f}x".format(a_linear, b_linear))
print("Exponential model: y = {:.3f} * e^({:.3f}x)".format(A, b_exp))
print("\nR^2 (linear):      {:.5f}".format(R2_linear))
print("R^2 (exponential): {:.5f}".format(R2_exp))

if R2_linear > R2_exp:
    print("\nThe linear model is the better fit according to the correlation coefficient used.")
else:
    print("\nThe exponential model is the better fit according to the correlation coefficient used.")

# Plotting
x_plot = np.linspace(min(x_vals), max(x_vals), 500)
y_linear_plot = linear_model(x_plot)
y_exp_plot = exponential_model(x_plot)

plt.figure(figsize=(10, 6))
plt.scatter(x_vals, y_vals, color='black', label="Data")
plt.plot(x_plot, y_linear_plot, label="Linear fit", color='blue')
plt.plot(x_plot, y_exp_plot, label="Exponential fit", color='red', linestyle='--')
plt.title("Comparison: Linear vs. Exponential Fit")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)
plt.savefig("curve_fitting_linear_vs_exponential.png", dpi=150, bbox_inches="tight")
plt.show()

### 5. Newton's divided-difference interpolation using the 4 nearest neighbors

In [ ]:
import numpy as np

# Original data
x_data = np.array([0.05, 0.4, 0.8, 1.2, 1.6, 2, 2.4])
y_data = np.array([550, 750, 1000, 1400, 2000, 2700, 3750])
x_target = 1.5

# Step 1: pick the 4 nearest neighbors to x = 1.5
distances = np.abs(x_data - x_target)
sorted_indices = np.argsort(distances)
neighbor_indices = np.sort(sorted_indices[:4])  # sort to keep x increasing

x_neighbors = x_data[neighbor_indices]
y_neighbors = y_data[neighbor_indices]

# Step 2: Newton interpolation using those 4 points
def newton_polynomial(x_points, y_points):
    n = len(x_points)
    coefficients = [0] * n
    diff_table = y_points.copy()

    for j in range(n):
        coefficients[j] = diff_table[0]
        for i in range(n - j - 1):
            diff_table[i] = (diff_table[i + 1] - diff_table[i]) / (x_points[i + j + 1] - x_points[i])

    def evaluate(x_value):
        result = coefficients[0]
        for i in range(1, n):
            term = coefficients[i]
            for j in range(i):
                term *= (x_value - x_points[j])
            result += term
        return result

    # Build the symbolic expression for P(x)
    expression = f"y = {coefficients[0]:.5f}"
    for i in range(1, n):
        term = f"{coefficients[i]:.5f}"
        for j in range(i):
            term += f"*(x - {x_points[j]:.2f})"
        expression += " + " + term
    return evaluate, expression

# Build and evaluate the Newton polynomial
evaluate_newton, newton_expression = newton_polynomial(x_neighbors, y_neighbors)
y_at_15 = evaluate_newton(1.5)

# Show the results
print("\n--- Cubic Newton polynomial based on the 4 nearest neighbors ---\n")
print("Selected points:")
for xi, yi in zip(x_neighbors, y_neighbors):
    print(f"({xi}, {yi})")
print("\nProposed polynomial in Newton form:\n")
print(newton_expression)
print(f"\nApproximate value at x = 1.5: P(1.5) = {y_at_15:.5f}")


### 6. Lagrange interpolation: P(4) and the polynomial used

In [ ]:
import numpy as np

# Ask the user for the data points
num_points = int(input("How many points do you have? "))

points_x = []
points_y = []

for index in range(num_points):
    x_value = float(input(f"Enter the value of x{index}: "))
    y_value = float(input(f"Enter the value of y{index}: "))
    points_x.append(x_value)
    points_y.append(y_value)

# ---------------- LAGRANGE POLYNOMIAL ----------------
def lagrange_polynomial(x_list, y_list):
    count = len(x_list)

    # Numerical evaluation of P(x)
    def evaluate(x_value):
        total_sum = 0
        for i in range(count):
            term = y_list[i]
            for j in range(count):
                if i != j:
                    term *= (x_value - x_list[j]) / (x_list[i] - x_list[j])
            total_sum += term
        return total_sum

    # Symbolic construction of P(x)
    polynomial_parts = []
    for i in range(count):
        term = f"{y_list[i]:.3f}"
        for j in range(count):
            if i != j:
                numerator = f"(x - {x_list[j]:.3f})"
                denominator = f"({x_list[i] - x_list[j]:.3f})"
                term += f" * {numerator}/{denominator}"
        polynomial_parts.append(term)

    expression = "P(x) = " + " + ".join(polynomial_parts)
    return evaluate, expression

# Build the Lagrange polynomial
evaluate_lagrange, lagrange_expression = lagrange_polynomial(points_x, points_y)

# Show the Lagrange polynomial
print("\n--- Lagrange polynomial constructed ---\n")
print(lagrange_expression)

# Compute and show the value of P(4)
value_at_4 = evaluate_lagrange(4)
print(f"\nThe polynomial evaluated at x = 4 is: P(4) = {value_at_4:.5f}")
